# Тернарные веса: от log₂(3) до обучающего шага

В этой лаборатории мы не используем NumPy, PyTorch и готовые упаковщики. Ожидаемый результат: пять весов из множества `{-1, 0, +1}` займут один байт, фактическая плотность кодов составит **1,6 бита на вес**, а учебный шаг с прямым оценивателем градиента уменьшит ошибку тернарного слоя.

## 1. Откуда берётся число 1,58

Один тернарный вес имеет три состояния. Число `log₂(3)` описывает размер этого алфавита в битах, но ещё не задаёт физическую раскладку в памяти.

In [ ]:
import math

information_bits = math.log2(3)
assert math.isclose(information_bits, 1.584962500721156)
assert 3 ** 5 <= 256 < 3 ** 6
print('информационных бит на трит:', information_bits)
print('состояний пяти тритов:', 3 ** 5)

## 2. Упаковка по основанию 3

Назначим цифры `-1 → 0`, `0 → 1`, `+1 → 2`. Пять цифр становятся разрядами одного числа по основанию 3. Функция возвращает также исходную длину: последний байт может содержать добавленные нулевые веса, которые при чтении нужно отбросить.

In [ ]:
def pack_five_trits(values):
    assert len(values) == 5
    assert all(value in (-1, 0, 1) for value in values)
    code = sum((value + 1) * (3 ** position)
               for position, value in enumerate(values))
    assert 0 <= code < 243
    return code

def unpack_five_trits(code):
    assert 0 <= code < 243
    values = []
    for _ in range(5):
        code, digit = divmod(code, 3)
        values.append(digit - 1)
    return values

example = [-1, 0, 1, -1, 1]
code = pack_five_trits(example)
assert code == 183
assert unpack_five_trits(code) == example
print('байт:', code, hex(code))

In [ ]:
def pack_trits(values):
    original_length = len(values)
    padded = list(values)
    padded.extend([0] * ((-original_length) % 5))
    data = bytes(pack_five_trits(padded[i:i + 5])
                 for i in range(0, len(padded), 5))
    return data, original_length

def unpack_trits(data, original_length):
    values = []
    for code in data:
        values.extend(unpack_five_trits(code))
    return values[:original_length]

weights = [(-1, 0, 1)[i % 3] for i in range(1003)]
packed, length = pack_trits(weights)
assert unpack_trits(packed, length) == weights
assert len(packed) == math.ceil(len(weights) / 5)
print('весов:', len(weights), 'байтов:', len(packed))

## 3. Логическая и физическая плотность

Сравним плотную запись пяти тритов в байте с простой раскладкой по два бита. Затем добавим один масштаб FP16 на группу из 128 весов. Это нижняя оценка: заголовок, формы, выравнивание и неполная последняя группа здесь не учитываются.

In [ ]:
n = 1280  # делится и на 5, и на размер группы
base3_bytes = n // 5
two_bit_bytes = n // 4
group_size = 128
scale_bits = 16

base3_bits_per_weight = 8 * base3_bytes / n
two_bit_bits_per_weight = 8 * two_bit_bytes / n
base3_with_scales = base3_bits_per_weight + scale_bits / group_size
two_bit_with_scales = two_bit_bits_per_weight + scale_bits / group_size

assert base3_bits_per_weight == 1.6
assert two_bit_bits_per_weight == 2.0
assert base3_with_scales == 1.725
assert two_bit_with_scales == 2.125
print(base3_bits_per_weight, base3_with_scales)
print(two_bit_bits_per_weight, two_bit_with_scales)

## 4. Квантование по среднему модулю

Сначала вычисляем `alpha = mean(abs(w))`, затем округляем `w / alpha` и ограничиваем результат уровнями `-1`, `0`, `+1`. Отдельно измеряем ошибку весов и ошибку выхода слоя: это разные величины.

In [ ]:
def round_away_from_zero(value):
    if value >= 0:
        return math.floor(value + 0.5)
    return math.ceil(value - 0.5)

def ternarize_absmean(values):
    assert values
    alpha = sum(abs(value) for value in values) / len(values)
    if alpha == 0:
        return 0.0, [0] * len(values)
    q = [max(-1, min(1, round_away_from_zero(value / alpha)))
         for value in values]
    restored = [alpha * value for value in q]
    return alpha, q, restored

w = [-0.9, -0.1, 0.2, 1.2]
alpha, q, w_hat = ternarize_absmean(w)
mse = sum((a - b) ** 2 for a, b in zip(w, w_hat)) / len(w)
assert math.isclose(alpha, 0.6)
assert q == [-1, 0, 0, 1]
print('alpha:', alpha, 'q:', q, 'MSE весов:', mse)

In [ ]:
def ternary_dot(q, x, alpha):
    accumulator = 0.0
    for weight, value in zip(q, x):
        if weight == 1:
            accumulator += value
        elif weight == -1:
            accumulator -= value
        else:
            assert weight == 0
    return alpha * accumulator

x = [0.5, 7.0, 1.5, -2.0]
reference = sum(weight * value for weight, value in zip(w, x))
quantized_output = ternary_dot(q, x, alpha)
assert math.isclose(quantized_output, -1.5)
print('выход до квантования:', reference)
print('выход после квантования:', quantized_output)

## 5. Главные веса и прямой оцениватель градиента

Тернарные коды не обновляются напрямую. Ниже `latent` — вещественные главные веса. Прямой проход использует их тернарную копию, а обратный проход приближённо передаёт производную так, словно квантователь был тождественным преобразованием. Для ясности мы игнорируем производную масштаба `alpha` внутри одного шага.

In [ ]:
latent = [0.20, 0.05, -0.15]
target = [1.20, 0.00, -1.20]  # ответы для трёх единичных входов
learning_rate = 0.05
loss_history = []

for step in range(500):
    alpha, q, quantized = ternarize_absmean(latent)
    loss = sum((prediction - answer) ** 2
               for prediction, answer in zip(quantized, target)) / len(target)
    loss_history.append(loss)

    # STE: d(quantized_i)/d(latent_i) приблизительно принимаем равной 1.
    gradients = [2 * (prediction - answer) / len(target)
                 for prediction, answer in zip(quantized, target)]
    latent = [weight - learning_rate * gradient
              for weight, gradient in zip(latent, gradients)]

final_alpha, final_q, final_quantized = ternarize_absmean(latent)
assert loss_history[-1] < loss_history[0] * 1e-6
assert final_q == [1, 0, -1]
assert all(abs(a - b) < 2e-5 for a, b in zip(final_quantized, target))
print('loss:', loss_history[0], '->', loss_history[-1])
print('главные веса:', latent)
print('тернарные коды:', final_q, 'alpha:', final_alpha)

## 6. Что именно доказал опыт

Мы воспроизвели четыре разных слоя механизма: размер трёхзначного алфавита, физическую упаковку, сложения вместо умножений и обучение скрытых вещественных весов через STE. Мы **не** доказали, что произвольную готовую BF16-модель можно без потерь превратить в тернарную: для этого нужен отдельный эксперимент обучения или QAT и сравнение качества на одном корпусе.

Задание на диагностику: поменяйте в `pack_five_trits` соответствие для `+1`, не меняя читатель. Какая проверка ломается первой и почему? Затем попробуйте сохранить шесть тритов одним байтом и объясните, на каком входе возникнет переполнение.